In [0]:
df = spark.read.table("gizmobox.bronze.py_orders")
df.display()

In [0]:
from pyspark.sql import functions as F
df_orders = df.select(F.regexp_replace("value", '"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date": "$1"').alias("fixed_value"))
df_orders.display()

####Transform JSON string to JSON object

In [0]:
df_JSON_Schema = df_orders.select(F.schema_of_json(F.col("fixed_value").alias("schema")))
df_JSON_Schema.display()

In [0]:
order_schema = '''STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>'''

In [0]:
df_JSON_orders = df_orders.select(F.from_json("fixed_value", order_schema).alias("json_value"))
df_JSON_orders.display()

In [0]:
df_orders_normalized = df_JSON_orders.select("json_value.order_id",
                                             "json_value.order_status",
                                             "json_value.payment_method",
                                             "json_value.total_amount",
                                             "json_value.transaction_timestamp",
                                             "json_value.customer_id",
                                             "json_value.items"
                                             
                                             )
df_orders_normalized.display()

####Deduplicate array elements

In [0]:
df_orders_normalized = df_JSON_orders.select("json_value.order_id",
                                             "json_value.order_status",
                                             "json_value.payment_method",
                                             "json_value.total_amount",
                                             "json_value.transaction_timestamp",
                                             "json_value.customer_id",
                                             F.array_distinct("json_value.items").alias("items")
                                             
                                             )
df_orders_normalized.display()

####Explode array

In [0]:
df_order_exploded = df_orders_normalized.select(
                                            "order_id",
                                            "order_status",
                                            "payment_method",
                                            "total_amount",
                                            "transaction_timestamp",
                                            "customer_id",
                                            F.explode("items").alias("item")
)
df_order_exploded.display()

In [0]:
df_orders = df_order_exploded.select(
                                             "order_id",
                                            "order_status",
                                            "payment_method",
                                            "total_amount",
                                            "transaction_timestamp",
                                            "customer_id",
                                            "item.item_id",
                                            "item.name",
                                            "item.price",
                                            "item.category",
                                            "item.details.brand",
                                            "item.details.color",
                                            "item.quantity"
                                            
)
df_orders.display()

In [0]:

df_orders.writeTo("gizmobox.silver.py_orders").createOrReplace()

In [0]:
%sql
select * from gizmobox.silver.py_orders;